In [8]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path
import matplotlib.patches as patches

# Generate 3 random shapes as separate files
for file_index in range(5):
    # Create a new figure for each image
    fig, ax = plt.subplots(figsize=(6, 6))

    # Generate grid points for a 4x4 grid
    grid_points = np.array([(x, y) for x in range(4) for y in range(4)])
    num_vertices = np.random.randint(4, 8)  # Random number of vertices between 4 and 7

    # Randomly select indices of points to use
    indices = np.random.choice(len(grid_points), size=num_vertices, replace=False)
    vertices = grid_points[indices]

    # Sort points to create a convex hull-like shape
    centroid = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - centroid[1], vertices[:, 0] - centroid[0])
    sorted_indices = np.argsort(angles)
    vertices = vertices[sorted_indices]

    # Create a closed path by adding the first point at the end
    vertices = np.vstack([vertices, vertices[0]])

    # Create a Path object
    codes = [Path.MOVETO] + [Path.LINETO] * (len(vertices) - 2) + [Path.CLOSEPOLY]
    path = Path(vertices, codes)

    # Add the patch to the axis with a random color
    color = np.random.rand(3,)  # Random RGB color
    patch = patches.PathPatch(path, facecolor='white', edgecolor='black', alpha=0.7, lw=2)
    ax.add_patch(patch)

    # Set the limits slightly larger than the grid
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, 3.5)

    # Remove axes, grid, and ticks for a clean look
    ax.set_axis_off()

    # Save the figure as a PNG file
    filename = f"{file_index}.png"
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.close(fig)  # Close the figure to free memory


In [19]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path
import matplotlib.patches as patches
import random
import os

# 第一部分：生成5张简单的封闭图形
def generate_simple_shape(image_index, grid_size=4, fill_shapes=False):
    """
    生成一个简单的封闭图形，并保存为PNG文件。

    Args:
        image_index: 图片的索引号
        grid_size: 网格大小
        fill_shapes: 是否填充图形颜色
    """
    # 创建新图像
    fig, ax = plt.subplots(figsize=(6, 6))

    # 设置网格范围
    ax.set_xlim(-0.5, grid_size-0.5)
    ax.set_ylim(-0.5, grid_size-0.5)

    # 不再绘制网格点

    # 生成随机封闭图形
    grid_points = np.array([(x, y) for x in range(grid_size) for y in range(grid_size)])
    num_vertices = np.random.randint(4, 8)
    indices = np.random.choice(len(grid_points), size=num_vertices, replace=False)
    vertices = grid_points[indices]

    # 排序点以创建类似凸包的形状
    centroid = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - centroid[1], vertices[:, 0] - centroid[0])
    sorted_indices = np.argsort(angles)
    vertices = vertices[sorted_indices]

    # 创建闭合路径
    vertices = np.vstack([vertices, vertices[0]])

    # 绘制形状
    if fill_shapes:
        # 如果需要填充，使用浅灰色填充
        codes = [Path.MOVETO] + [Path.LINETO] * (len(vertices) - 2) + [Path.CLOSEPOLY]
        path = Path(vertices, codes)
        patch = patches.PathPatch(path, facecolor='lightgray', edgecolor='black', linewidth=2)
        ax.add_patch(patch)
    else:
        # 不填充，只画线
        for i in range(len(vertices)-1):
            ax.plot([vertices[i][0], vertices[i+1][0]],
                    [vertices[i][1], vertices[i+1][1]],
                    'k-', linewidth=2)

    # 移除坐标轴
    ax.set_axis_off()

    # 保存图像
    plt.savefig(f'0-{image_index}.png', bbox_inches='tight', dpi=300)
    plt.close()

    return vertices

# 创建一个随机封闭图形的函数
def create_random_closed_shape(grid_size):
    """
    在给定网格大小内创建一个随机封闭图形

    Args:
        grid_size: 网格大小

    Returns:
        vertices: 图形顶点的numpy数组
    """
    # 随机选择3-6个网格点
    num_points = random.randint(3, 6)
    points = []
    for _ in range(num_points):
        x = random.randint(0, grid_size-1)
        y = random.randint(0, grid_size-1)
        points.append((x, y))

    # 确保至少有3个不同的点
    unique_points = list(set(points))
    if len(unique_points) < 3:
        # 如果不足3个点，添加额外的点
        while len(unique_points) < 3:
            x = random.randint(0, grid_size-1)
            y = random.randint(0, grid_size-1)
            new_point = (x, y)
            if new_point not in unique_points:
                unique_points.append(new_point)

    # 转换为numpy数组
    vertices = np.array(unique_points)

    # 排序点以创建类似凸包的形状
    centroid = np.mean(vertices, axis=0)
    angles = np.arctan2(vertices[:, 1] - centroid[1], vertices[:, 0] - centroid[0])
    sorted_indices = np.argsort(angles)
    vertices = vertices[sorted_indices]

    # 创建闭合路径
    vertices = np.vstack([vertices, vertices[0]])

    return vertices

# 第二部分：生成复杂图形
def generate_complex_image(image_index, simple_shapes, num_original_images=5, grid_size=6, fill_original_shapes=False):
    """
    生成一个复杂图片，将随机选择的原始图片放置在grid_size×grid_size的网格上，
    并添加干扰的封闭图形。

    Args:
        image_index: 复杂图片的索引号
        simple_shapes: 简单图形的顶点列表
        num_original_images: 原始图片的总数
        grid_size: 网格大小
        fill_original_shapes: 是否填充原始图形颜色
    """
    # 创建新图像
    fig, ax = plt.subplots(figsize=(8, 8))

    # 设置网格范围
    ax.set_xlim(-0.5, grid_size-0.5)
    ax.set_ylim(-0.5, grid_size-0.5)

    # 不再绘制网格点

    # 决定使用多少个原始图片 (1到3之间的随机数)
    num_images_to_include = random.randint(1, min(num_original_images, 3))

    # 随机选择要包含的图片
    available_images = list(range(num_original_images))
    selected_images = random.sample(available_images, num_images_to_include)

    # 跟踪哪些图片被使用
    image_used = {i: False for i in range(num_original_images)}

    # 处理每个选定的图片
    for img_idx in selected_images:
        image_used[img_idx] = True

        # 获取简单图形的顶点
        vertices = simple_shapes[img_idx]

        # 随机放置形状在网格上，但保持整齐
        max_offset = grid_size - 4  # 确保4×4的形状适合在grid_size×grid_size的网格中
        offset_x = random.randint(0, max_offset)
        offset_y = random.randint(0, max_offset)

        # 应用偏移
        shifted_vertices = vertices.copy()
        shifted_vertices[:, 0] += offset_x
        shifted_vertices[:, 1] += offset_y

        # 绘制形状 - 原始图形可以选择填充
        if fill_original_shapes:
            # 如果需要填充，使用浅灰色填充
            codes = [Path.MOVETO] + [Path.LINETO] * (len(shifted_vertices) - 2) + [Path.CLOSEPOLY]
            path = Path(shifted_vertices, codes)
            patch = patches.PathPatch(path, facecolor='lightgray', edgecolor='black', linewidth=2)
            ax.add_patch(patch)
        else:
            # 不填充，只画线
            for i in range(len(shifted_vertices)-1):
                ax.plot([shifted_vertices[i][0], shifted_vertices[i+1][0]],
                        [shifted_vertices[i][1], shifted_vertices[i+1][1]],
                        'k-', linewidth=2)

    # 添加随机封闭图形作为干扰 (5-10个) - 干扰图形不填充
    num_distraction_shapes = random.randint(5, 10)
    for _ in range(num_distraction_shapes):
        # 创建随机封闭图形
        vertices = create_random_closed_shape(grid_size)

        # 绘制形状 - 干扰图形永远不填充
        for i in range(len(vertices)-1):
            ax.plot([vertices[i][0], vertices[i+1][0]],
                    [vertices[i][1], vertices[i+1][1]],
                    'k-', linewidth=2)

    # 移除坐标轴
    ax.set_axis_off()

    # 保存最终图像
    plt.savefig(f'{image_index}.png', bbox_inches='tight', dpi=300)
    plt.close()

    # 创建对应的txt文件，记录哪些原始图片被使用
    with open(f'{image_index}.txt', 'w') as f:
        for i in range(num_original_images):
            if image_used[i]:
                f.write("yes\n")
            else:
                f.write("no\n")

    return image_used

# 主程序
def main():
    print("第一步：生成5个简单的封闭图形...")
    simple_shapes = []
    for i in range(5):
        vertices = generate_simple_shape(i, fill_shapes=False)
        simple_shapes.append(vertices)
        print(f"生成了简单图形 0-{i}.png")

    print("\n第二步：生成复杂图形...")
    num_complex_images = int(input("请输入要生成的复杂图片数量: "))

    for i in range(1, num_complex_images+1):
        used_images = generate_complex_image(i, simple_shapes, fill_original_shapes=False)
        print(f"生成了{i}.png和{i}.txt")
        print(f"使用的原始图片: {[idx for idx, used in used_images.items() if used]}")

if __name__ == "__main__":
    main()


第一步：生成5个简单的封闭图形...
生成了简单图形 0-0.png
生成了简单图形 0-1.png
生成了简单图形 0-2.png
生成了简单图形 0-3.png
生成了简单图形 0-4.png

第二步：生成复杂图形...
生成了1.png和1.txt
使用的原始图片: [0, 3]
生成了2.png和2.txt
使用的原始图片: [0, 2, 3]
生成了3.png和3.txt
使用的原始图片: [3]
生成了4.png和4.txt
使用的原始图片: [0, 2, 4]
生成了5.png和5.txt
使用的原始图片: [0]
